# Figure 1: exact peptide lookup selects compact search databases

Panel a is a schematic of the database strategies and of one worked healthy-stool acquisition; it ships as artwork and its typed numbers are recomputed here. Panels b to e are regenerated from `source_data/Fig1/` with the plot code of the original producers and compared pixel by pixel with the panels those producers rendered. The text statistics of Results lines 51 to 65 are recomputed in the last cell.

Run from the repository root: `python paper.py build Fig1`.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

In [ ]:
D = panels.load(SD)

## Panel a

Schematic (v33). Human proteomics, conventional metaproteomics and FastaLake, with the worked acquisition `20181231_QX7_PhGe_LC95_SA_WorkflowReplicates_2` and its composition-matched scrambled control (seed 20260901). Supports "FastaLake pools de novo peptide queries from the 58 healthy-stool acquisitions ... shown for one acquisition" (legend, v28d line 71) and Results line 53. n: one acquisition; the study-level lookup pools 58 acquisitions. No test. The FastaLake column (one funnel: reservoir, exact match to study-supported proteins, per-acquisition match, parsimony) is rebuilt by `artwork/build_fig1a_funnel.py` (v33 layout) in the last cell and compared with `expected/Fig1_v33.png`. Every typed number is recomputed below and traced in `ARTWORK_NUMBERS.md`.

In [ ]:
panels.show(panels.panel_a(UNIT, OUT))

In [ ]:
for k, v in panels.artwork_numbers(D).items():
    print(f"{k:60s} {v}")

## Panel b

Entries in each acquisition-specific FASTA, by cohort; dashed line, the 20,431-entry human reference proteome. Supports "All 726 acquisition-specific databases contained fewer entries than the 20,431-entry human reference proteome, with median sizes ranging from 3,100 proteins in CAMPI to 12,200 in healthy stool" (v28d line 55). n = 726 acquisitions (15 CAMPI, 58 healthy stool, 306 PREDICT, 347 CapScan). Descriptive; bars are medians. No test.

In [ ]:
panels.panel_b(D, OUT)
print(panels.stat_b_sizes(D))
panels.show(OUT / "panel_b.png")

## Panel c

Fraction of distinct authentic or scrambled queries of the worked acquisition with an exact match in the 233,497,702-sequence reservoir, by peptide length; five scrambling seeds (thin lines) and their median. Supports "The fraction of scrambled queries matching a reservoir protein fell from 8.1–8.9% for nine-residue queries to 0.7–1.1% at ten residues, with no match in 86,000 query–seed evaluations at ≥13 residues" (v28d line 59). Unit: distinct query within one seed; n per seed and length in `length_by_arm.tsv`. Range over seeds; no test.

In [ ]:
panels.panel_c(D, OUT)
print(panels.stat_c_scramble(D))
panels.show(OUT / "panel_c.png")

## Panel d

Per acquisition, the fraction of selected proteins that contain exactly one distinct production (top-30%) lookup query. Supports "Across cohorts, a median 21.6–35.8% of selected proteins matched only one distinct top-scoring lookup query" and "95.7–97.7% of selected proteins contained at least one qualifying prediction whose sequence the final search also identified" (v28d line 63). n = 726 acquisitions as in panel b. Bars are medians. No test. The 4,258 of 29,663 accepted peptides absent from the archived predictions (same line) is computed here too.

In [ ]:
panels.panel_d(D, OUT)
print(panels.stat_d_support(D))
print(panels.stat_d_not_predicted(D))
panels.show(OUT / "panel_d.png")

## Panel e

Donor-selected entrapment estimates (FDRBench paired estimator) in 58 healthy-stool acquisitions, each the mean of three entrapment seeds, for accepted peptides of 5–50 and 9–50 residues at peptide q ≤ 0.01. Supports "The median paired estimate was 1.0% for accepted 5–50-residue peptides and 0.49% for the 9–50-residue subset of the same searches" (v28d line 65). Unit: acquisition, n = 58. Bars are medians. No test. The seed means are recomputed from the per-seed counts.

In [ ]:
panels.panel_e(D, OUT)
print(panels.stat_e_entrapment(D))
panels.show(OUT / "panel_e.png")

## Checks and numbers

Pixel checks of panels b to e against the original producers' renders (`expected/`), a check that `expected/panel_e.png` is the stated crop of the copied v10 render, a check that `expected/Fig1_v28f.png` is the 300 dpi raster of the v28f composite PDF (the base page of the funnel build), a check that `expected/Fig1_v33.png` (the Figure 1 of manuscript v33) is the 300 dpi raster of `artwork/Figure1_v33.pdf`, the v33 funnel rebuilt from `artwork/` and compared with `expected/Fig1_v33.png`, and every recomputed number in `NUMBERS.tsv` order.

In [ ]:
import csv, os, re
from tools.check_pixels import compare
CHK = UNIT / "checks"
if not os.environ.get("FASTALAKE_FONT_DIR") and not os.environ.get("FL_STYLE_DIR"):
    print("pixel checks SKIPPED: fonts absent")
else:
    E = UNIT / "expected"
    fig, axs, _ = panels._row(D)
    panels.crop_panel_e(E / "panels_bcde_row_v10.pdf", OUT / "panel_e_expected_rederived.png", fig, axs)
    jobs = [("panel_b", OUT / "panel_b.pdf", E / "panel_b.pdf"),
            ("panel_c", OUT / "panel_c.pdf", E / "panel_c.pdf"),
            ("panel_d", OUT / "panel_d.pdf", E / "panel_d.pdf"),
            ("panel_e", OUT / "panel_e.png", E / "panel_e.png"),
            ("panel_e_expected_derivation", OUT / "panel_e_expected_rederived.png", E / "panel_e.png"),
            ("Fig1_v28f", UNIT / "artwork" / "Figure1_v28f.pdf", E / "Fig1_v28f.png"),
            ("Fig1_v33", UNIT / "artwork" / "Figure1_v33.pdf", E / "Fig1_v33.png"),
            ("Fig1_v33_rebuild", panels.build_panel_a_funnel(UNIT, OUT), E / "Fig1_v33.png")]
    for name, new, exp in jobs:
        r = compare(new, exp, CHK / name, dpi=panels.DPI)
        print(f"{name:30s} {r['verdict']:14s} differing pixels {r.get('differing_pixels')}")
with open(UNIT / "NUMBERS.tsv", encoding="utf-8") as fh:
    rows = list(csv.DictReader(fh, delimiter="\t"))
V = panels.numbers(D)
for r in rows:
    key = re.search(r"\(key (\w+)\)", r["code"]).group(1)
    print(f"line {r['v28d_line']:>3}  printed {r['printed_value']:>12}  recomputed {V[key]!r}")